# Group Preprocessing Demonstration

This notebook consolidates the six individual Progress Review I contributions: data quality, outliers, categorical encoding, feature selection, standardization and PCA.

**Scope:** An explanatory exercise on the 80,000 development rows. It produces an in-memory demonstration table, not the training input for the six headline models. No processed employee dataset or fitted transformers are exported.

**Model integration:** Each member notebook uses its model implementation in `src/models/` to prepare inputs within the appropriate training partition. Canonical model predictions are registered in `configs/analysis_registry.yaml`.

**Validation boundary:** Holdout observations are excluded from the transformations here. The demonstration fits the encoder, scaler and PCA on all development rows, including development validation folds. Its transformed table must not be used as a precomputed input for cross-validation: fit preprocessing separately within each training fold. These demonstration transformers are not connected to the final holdout notebook.

In [1]:
# Cell 2: Imports and Environment Setup
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.decomposition import PCA

# Formatting and display options
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")
plt.rcParams["figure.dpi"] = 120
print("Environment initialized with all required libraries.")

Environment initialized with all required libraries.


In [2]:
# Cell 3: Robust Path Resolution & Development Partition Loading
def find_project_root():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "configs" / "feature_contract.yaml").is_file():
            return candidate
    raise FileNotFoundError("Project root containing configs/feature_contract.yaml not found")

PROJECT_ROOT = find_project_root()

def resolve_path(rel):
    return PROJECT_ROOT / rel

DATA_PATH = resolve_path("data/raw/Extended_Employee_Performance_and_Productivity_Data.csv")
SPLIT_PATH = resolve_path("data/processed/split_registry.csv")
CONTRACT_PATH = resolve_path("configs/feature_contract.yaml")

# Load raw data and frozen split registry
df_raw = pd.read_csv(DATA_PATH)
split_registry = pd.read_csv(SPLIT_PATH)

print(f"Raw dataset shape: {df_raw.shape}")
print(f"Split registry shape: {split_registry.shape}")
print("Registry partition counts:")
print(split_registry['partition'].value_counts().to_string())

# Restrict strictly to development partition (80,000 rows)
dev_filter = split_registry["partition"] == "development"
df_current = df_raw.merge(
    split_registry.loc[dev_filter, ["Employee_ID"]],
    on="Employee_ID",
    how="inner",
    validate="one_to_one"
).reset_index(drop=True)

assert len(df_current) == 80_000, f"Expected 80,000 development rows, found {len(df_current)}"
assert df_current["Employee_ID"].is_unique, "Primary key Employee_ID must be unique!"

print(f"\n[Step 0 Initialized] Development partition shape: {df_current.shape}")
print(f"Columns ({df_current.shape[1]}): {df_current.columns.tolist()}")

assert set(df_current["Employee_ID"]).isdisjoint(
    set(split_registry.loc[split_registry["partition"] == "holdout", "Employee_ID"])
)
print("Development target counts:")
print(df_current["Resigned"].astype(int).value_counts().sort_index().to_string())
print(f"Resigned prevalence: {df_current['Resigned'].mean():.2%}")
print("Class imbalance: no resampling in this demonstration; weighting is handled by the model workflows.")


Raw dataset shape: (100000, 20)
Split registry shape: (100000, 3)
Registry partition counts:
partition
development    80000
holdout        20000

[Step 0 Initialized] Development partition shape: (80000, 20)
Columns (20): ['Employee_ID', 'Department', 'Gender', 'Age', 'Job_Title', 'Hire_Date', 'Years_At_Company', 'Education_Level', 'Performance_Score', 'Monthly_Salary', 'Work_Hours_Per_Week', 'Projects_Handled', 'Overtime_Hours', 'Sick_Days', 'Remote_Work_Frequency', 'Team_Size', 'Training_Hours', 'Promotions', 'Employee_Satisfaction_Score', 'Resigned']
Development target counts:
Resigned
0    71992
1     8008
Resigned prevalence: 10.01%
Class imbalance: no resampling in this demonstration; weighting is handled by the model workflows.


## Step 1: Data Quality Validation & Missing Value Handling
**Sourced from:** `notebooks/team_contributions/preprocessing/IT25100565_Data_Quality.ipynb` (IT25100565)

### Methodology & Decision:
- **Completeness Audit:** Evaluates missingness across all 20 features (`isnull().sum()`).
- **Duplicate Detection:** Audits full-row duplicates across all features (`duplicated().sum()`).
- **Primary Key Integrity:** Verifies unique 1-to-1 mapping for `Employee_ID`.
- **Handling Decision:** In `IT25100565_Data_Quality.ipynb`, the audit confirmed **0 missing values** and **0 duplicate rows** across the entire dataset. Therefore, **no rows were dropped and no artificial imputation (mean/median/mode/k-NN) was applied**, preserving the statistical integrity of the original observations.

In [3]:
# Step 1 Code: Data Quality Audit
null_counts = df_current.isnull().sum()
total_nulls = int(null_counts.sum())
full_duplicates = int(df_current.duplicated().sum())
unique_ids = int(df_current["Employee_ID"].nunique())

print("--- Step 1 Audit Results ---")
print(f"Total missing values across all cells: {total_nulls}")
print(f"Total duplicate rows detected: {full_duplicates}")
print(f"Unique Employee_ID count: {unique_ids:,} (Integrity verified: {unique_ids == len(df_current)})")

assert total_nulls == 0, "Unexpected missing values found in development data!"
assert full_duplicates == 0, "Duplicate rows detected in development data!"
assert unique_ids == len(df_current) == 80_000

print(f"\nShape after Step 1: {df_current.shape}")
print(f"Columns ({df_current.shape[1]}): {df_current.columns.tolist()}")

--- Step 1 Audit Results ---
Total missing values across all cells: 0
Total duplicate rows detected: 0
Unique Employee_ID count: 80,000 (Integrity verified: True)

Shape after Step 1: (80000, 20)
Columns (20): ['Employee_ID', 'Department', 'Gender', 'Age', 'Job_Title', 'Hire_Date', 'Years_At_Company', 'Education_Level', 'Performance_Score', 'Monthly_Salary', 'Work_Hours_Per_Week', 'Projects_Handled', 'Overtime_Hours', 'Sick_Days', 'Remote_Work_Frequency', 'Team_Size', 'Training_Hours', 'Promotions', 'Employee_Satisfaction_Score', 'Resigned']


## Step 2: Outlier Detection & Handling (1.5 $\times$ IQR Rule)
**Sourced from:** `notebooks/team_contributions/preprocessing/IT24100879_Outlier_Analysis.ipynb` (IT24100879)

### Methodology & Decision:
- **IQR Boundary Calculation:** For each of the 13 approved numerical predictors, computes:
  $$\text{IQR} = Q_3 - Q_1, \quad \text{Lower Fence} = Q_1 - 1.5 \times \text{IQR}, \quad \text{Upper Fence} = Q_3 + 1.5 \times \text{IQR}$$
- **Evaluation:** The table reports observed flags; an IQR flag alone does not establish data corruption.
- **Handling Decision:** **All 80,000 rows are retained without dropping, clipping, or replacement.** As defended in `IT24100879_Outlier_Analysis.ipynb`, statistical unusualness represents valid workforce diversity (e.g., highly promoted employees, heavy overtime contributors), not data corruption. Deleting or truncating records would distort the genuine distribution and compromise real-world predictive validity.

In [4]:
# Step 2 Code: IQR Outlier Detection and Fence Computation
with open(CONTRACT_PATH, "r") as f:
    contract_yaml = yaml.safe_load(f)

num_cols = [feature["name"] for feature in contract_yaml["numerical_features"]]
assert len(num_cols) == 13, f"Expected 13 numerical features, got {len(num_cols)}"

q1 = df_current[num_cols].quantile(0.25)
q3 = df_current[num_cols].quantile(0.75)
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr

outlier_flags = (df_current[num_cols] < lower_fence) | (df_current[num_cols] > upper_fence)
flagged_per_feature = outlier_flags.sum()

outlier_summary = pd.DataFrame({
    "Q1 (25%)": q1,
    "Q3 (75%)": q3,
    "IQR": iqr,
    "Lower Fence": lower_fence,
    "Upper Fence": upper_fence,
    "Flagged Values": flagged_per_feature,
    "Flagged (%)": (flagged_per_feature / len(df_current)) * 100
})

print("--- Step 2 IQR Boundary Analysis ---")
display(outlier_summary)

# Decision: Retain all 80,000 rows without removal or capping
assert len(df_current) == 80_000, "Row count changed unexpectedly!"
print(f"Decision: All {len(df_current):,} rows retained (0 rows dropped, no capping applied).")

print(f"\nShape after Step 2: {df_current.shape}")
print(f"Columns ({df_current.shape[1]}): {df_current.columns.tolist()}")

--- Step 2 IQR Boundary Analysis ---


,Q1 (25%),Q3 (75%),IQR,Lower Fence,Upper Fence,Flagged Values,Flagged (%)
Age,31.0000,51.0000,20.0000,1.0000,81.0000,0,0.0000
Years_At_Company,2.0000,7.0000,5.0000,-5.5000,14.5000,0,0.0000
Performance_Score,2.0000,4.0000,2.0000,-1.0000,7.0000,0,0.0000
Monthly_Salary,5250.0000,7500.0000,2250.0000,1875.0000,10875.0000,0,0.0000
Work_Hours_Per_Week,37.0000,53.0000,16.0000,13.0000,77.0000,0,0.0000
Projects_Handled,12.0000,37.0000,25.0000,-25.5000,74.5000,0,0.0000
Overtime_Hours,7.0000,22.0000,15.0000,-15.5000,44.5000,0,0.0000
Sick_Days,3.0000,11.0000,8.0000,-9.0000,23.0000,0,0.0000
Remote_Work_Frequency,25.0000,75.0000,50.0000,-50.0000,150.0000,0,0.0000
Team_Size,5.0000,15.0000,10.0000,-10.0000,30.0000,0,0.0000


Decision: All 80,000 rows retained (0 rows dropped, no capping applied).

Shape after Step 2: (80000, 20)
Columns (20): ['Employee_ID', 'Department', 'Gender', 'Age', 'Job_Title', 'Hire_Date', 'Years_At_Company', 'Education_Level', 'Performance_Score', 'Monthly_Salary', 'Work_Hours_Per_Week', 'Projects_Handled', 'Overtime_Hours', 'Sick_Days', 'Remote_Work_Frequency', 'Team_Size', 'Training_Hours', 'Promotions', 'Employee_Satisfaction_Score', 'Resigned']


## Step 3: Categorical Encoding (OneHotEncoder)
**Sourced from:** `notebooks/team_contributions/preprocessing/IT25100385_Categorical_Encoding.ipynb` (IT25100385)

### Methodology & Decision:
- **Approved Nominal Predictors:** `Department`, `Gender`, `Job_Title`, `Education_Level` (4 columns).
- **Encoder Choice:** `OneHotEncoder(sparse_output=False, handle_unknown="ignore")`.
- **Rationale:** Nominal categories possess no intrinsic numerical order. One-hot encoding creates orthogonal binary flags without imposing artificial ordinal distances (which label encoding would incorrectly do). `handle_unknown="ignore"` ensures future unseen categories do not trigger runtime errors.
- **Dimensionality Change:** The 4 original categorical features expand into **23 binary indicator columns** ($0/1$).
- **Shape Impact:** $20 - 4 + 23 = \mathbf{39}$ columns. Rows remain at 80,000.

In [5]:
# Step 3 Code: Fit OneHotEncoder on Development Partition
cat_cols = [feature["name"] for feature in contract_yaml["categorical_features"]]
assert cat_cols == ["Department", "Gender", "Job_Title", "Education_Level"]

encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
encoded_cat_array = encoder.fit_transform(df_current[cat_cols])
encoded_cat_names = encoder.get_feature_names_out(cat_cols).tolist()

assert len(encoded_cat_names) == 23, f"Expected 23 encoded categories, got {len(encoded_cat_names)}"

df_encoded_cats = pd.DataFrame(
    encoded_cat_array,
    columns=encoded_cat_names,
    index=df_current.index
)

print(f"Shape before encoding: {df_current.shape}")

# Replace the 4 original categorical columns with the 23 encoded one-hot columns
non_cat_cols = [c for c in df_current.columns if c not in cat_cols]
df_current = pd.concat([df_current[non_cat_cols], df_encoded_cats], axis=1)

assert df_current.shape == (80_000, 39), f"Unexpected shape after encoding: {df_current.shape}"


print(f"\nShape after Step 3: {df_current.shape}")
print(f"Original categorical columns dropped (4): {cat_cols}")
print(f"Encoded one-hot columns added ({len(encoded_cat_names)}): {encoded_cat_names}")
print(f"Total columns now: {df_current.shape[1]}")


Shape before encoding: (80000, 20)

Shape after Step 3: (80000, 39)
Original categorical columns dropped (4): ['Department', 'Gender', 'Job_Title', 'Education_Level']
Encoded one-hot columns added (23): ['Department_Customer Support', 'Department_Engineering', 'Department_Finance', 'Department_HR', 'Department_IT', 'Department_Legal', 'Department_Marketing', 'Department_Operations', 'Department_Sales', 'Gender_Female', 'Gender_Male', 'Gender_Other', 'Job_Title_Analyst', 'Job_Title_Consultant', 'Job_Title_Developer', 'Job_Title_Engineer', 'Job_Title_Manager', 'Job_Title_Specialist', 'Job_Title_Technician', 'Education_Level_Bachelor', 'Education_Level_High School', 'Education_Level_Master', 'Education_Level_PhD']
Total columns now: 39


## Step 4: Feature Selection & Redundancy Filtering
**Sourced from:** `notebooks/team_contributions/preprocessing/IT25100445_Feature_Selection.ipynb` (IT25100445)

### Methodology & Decision:
- **Identifier Exclusion:** `Employee_ID` is an arbitrary surrogate primary key. It is excluded to avoid learning arbitrary row-order patterns; it is **dropped from the predictor set**.
- **Tenure Redundancy Removal:** `Hire_Date` is a raw timestamp. The audit in `IT25100445_Feature_Selection.ipynb` showed **99.27% exact agreement** and **100% agreement within 1 year** with `Years_At_Company`. Dropping `Hire_Date` avoids a redundant date representation while preserving `Years_At_Company` as the official tenure predictor.
- **Multicollinearity Check:** Pearson correlation on numerical predictors confirmed the maximum correlation is $r = 0.5100$ (`Performance_Score` $\leftrightarrow$ `Monthly_Salary`), well within acceptable collinearity bounds ($|r| < 0.85$). All 13 numerical predictors are retained.
- **Target Separation:** `Resigned` is isolated as the prediction target $y$.
- **Shape Impact:** Dropping `Employee_ID` and `Hire_Date` reduces columns from 39 to **37** ($39 - 2 = 37$: 13 numerical predictors + 23 one-hot categorical predictors + 1 target `Resigned`).

In [6]:
# Step 4 Code: Drop Non-Predictive & Redundant Columns
dropped_features = ["Employee_ID", "Hire_Date"]
target_col = "Resigned"

for col in dropped_features:
    assert col in df_current.columns, f"Column {col} missing from dataframe"
assert target_col in df_current.columns, f"Target {target_col} missing from dataframe"

print(f"Shape before feature selection: {df_current.shape}")
df_current = df_current.drop(columns=dropped_features)
assert df_current.shape == (80_000, 37), f"Unexpected shape after feature selection: {df_current.shape}"

retained_predictors = [c for c in df_current.columns if c != target_col]
assert len(retained_predictors) == 36, f"Expected 36 predictors, got {len(retained_predictors)}"

print(f"\nShape after Step 4: {df_current.shape}")
print(f"Columns dropped: {dropped_features}")
print(f"Retained columns ({df_current.shape[1]}): {df_current.columns.tolist()}")

Shape before feature selection: (80000, 39)

Shape after Step 4: (80000, 37)
Columns dropped: ['Employee_ID', 'Hire_Date']
Retained columns (37): ['Age', 'Years_At_Company', 'Performance_Score', 'Monthly_Salary', 'Work_Hours_Per_Week', 'Projects_Handled', 'Overtime_Hours', 'Sick_Days', 'Remote_Work_Frequency', 'Team_Size', 'Training_Hours', 'Promotions', 'Employee_Satisfaction_Score', 'Resigned', 'Department_Customer Support', 'Department_Engineering', 'Department_Finance', 'Department_HR', 'Department_IT', 'Department_Legal', 'Department_Marketing', 'Department_Operations', 'Department_Sales', 'Gender_Female', 'Gender_Male', 'Gender_Other', 'Job_Title_Analyst', 'Job_Title_Consultant', 'Job_Title_Developer', 'Job_Title_Engineer', 'Job_Title_Manager', 'Job_Title_Specialist', 'Job_Title_Technician', 'Education_Level_Bachelor', 'Education_Level_High School', 'Education_Level_Master', 'Education_Level_PhD']


## Step 5: Numerical Feature Standardization (StandardScaler)
**Sourced from:** `notebooks/team_contributions/preprocessing/IT25100429_Standardization.ipynb` (IT25100429)

### Methodology & Decision:
- **Magnitude Imbalance:** Raw numerical features operate on drastically different ranges (e.g. `Monthly_Salary` with variance $\approx 1.88 \times 10^6$ vs. `Promotions` with variance $\approx 0.67$).
- **Method:** `StandardScaler()`, applying $z = (x - \mu)/\sigma$.
- **Fitting Scope:** Fitted strictly on the 13 numerical predictors of the development partition. Binary one-hot columns and the target `Resigned` are not standardized.
- **Verification:** Empirically verifies mean $\approx 0$ and population standard deviation $\sigma = 1.0000$ ($\text{ddof}=0$).
- **Shape Impact:** Shape remains **(80,000, 37)** (in-place scale transformation of the 13 numerical features).

In [7]:
# Step 5 Code: Fit and Apply StandardScaler on Numerical Predictors
scaler = StandardScaler()
scaled_num_array = scaler.fit_transform(df_current[num_cols])

df_scaled_num = pd.DataFrame(
    scaled_num_array,
    columns=num_cols,
    index=df_current.index
)

# Update the numerical columns in df_current with their standardized values
df_current[num_cols] = df_scaled_num

# Verification of standardization statistics
means = df_current[num_cols].mean().to_numpy()
stds = df_current[num_cols].std(ddof=0).to_numpy()

assert np.allclose(means, 0.0, atol=1e-10), "Standardized means are not zero!"
assert np.allclose(stds, 1.0, atol=1e-10), "Standardized population standard deviations are not 1.0!"
print("Standardization verified: All 13 numerical predictors have mean ~ 0 and population std == 1.0000.")


print(f"\nShape after Step 5: {df_current.shape}")
print(f"Standardized numerical columns ({len(num_cols)}): {num_cols}")
print(f"Total columns: {df_current.shape[1]}")


Standardization verified: All 13 numerical predictors have mean ~ 0 and population std == 1.0000.

Shape after Step 5: (80000, 37)
Standardized numerical columns (13): ['Age', 'Years_At_Company', 'Performance_Score', 'Monthly_Salary', 'Work_Hours_Per_Week', 'Projects_Handled', 'Overtime_Hours', 'Sick_Days', 'Remote_Work_Frequency', 'Team_Size', 'Training_Hours', 'Promotions', 'Employee_Satisfaction_Score']
Total columns: 37


## Step 6: PCA Dimensionality Reduction
**Sourced from:** `notebooks/team_contributions/preprocessing/IT25100142_PCA.ipynb` (IT25100142)

### Methodology & Decision:
- **Variance Analysis:** As proven in `IT25100142_PCA.ipynb`, unscaled PCA would be completely dominated by `Monthly_Salary`. Standardization in Step 5 converted the covariance matrix to the correlation matrix, enabling fair variance extraction.
- **Component Selection:** Cumulative explained variance confirms:
  - 10 PCs $\rightarrow \approx 81.08\%$
  - 11 PCs $\rightarrow \approx 88.67\%$
  - 12 PCs $\rightarrow \approx 96.23\%$
  - **Selected Threshold:** $\ge 95\%$ variance requires **12 Principal Components** (`PC1` to `PC12`).
- **Transformation:** Replaces the 13 original standardized numerical predictors with the 12 orthogonal Principal Components.
- **Shape Impact:** $37 - 13 + 12 = \mathbf{36}$ columns (12 PCs + 23 one-hot categories + 1 target `Resigned`). Rows: 80,000.

In [8]:
# Step 6 Code: Fit PCA and Transform Standardized Numerical Features
pca = PCA(n_components=12)
pca_transformed = pca.fit_transform(df_current[num_cols])
cumulative_variance = np.cumsum(pca.explained_variance_ratio_)
assert cumulative_variance[-1] >= 0.95
print(f"Shape before PCA: {df_current.shape}")

print(f"PCA Fitted: 12 components explain {cumulative_variance[-1] * 100:.2f}% of numerical variance (>= 95% threshold).")

pc_names = [f"PC{i+1}" for i in range(12)]
df_pca = pd.DataFrame(pca_transformed, columns=pc_names, index=df_current.index)

# Replace 13 numerical columns with 12 Principal Components
other_cols = [c for c in df_current.columns if c not in num_cols]
df_preprocessed = pd.concat([df_pca, df_current[other_cols]], axis=1)

assert df_preprocessed.shape == (80_000, 36), f"Unexpected final preprocessed shape: {df_preprocessed.shape}"
assert "Resigned" in df_preprocessed.columns, "Target column Resigned missing from preprocessed dataset!"


print(f"\nShape after Step 6 (Final Preprocessed): {df_preprocessed.shape}")
print(f"Columns ({df_preprocessed.shape[1]}): {df_preprocessed.columns.tolist()}")



Shape before PCA: (80000, 37)
PCA Fitted: 12 components explain 96.23% of numerical variance (>= 95% threshold).

Shape after Step 6 (Final Preprocessed): (80000, 36)
Columns (36): ['PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9', 'PC10', 'PC11', 'PC12', 'Resigned', 'Department_Customer Support', 'Department_Engineering', 'Department_Finance', 'Department_HR', 'Department_IT', 'Department_Legal', 'Department_Marketing', 'Department_Operations', 'Department_Sales', 'Gender_Female', 'Gender_Male', 'Gender_Other', 'Job_Title_Analyst', 'Job_Title_Consultant', 'Job_Title_Developer', 'Job_Title_Engineer', 'Job_Title_Manager', 'Job_Title_Specialist', 'Job_Title_Technician', 'Education_Level_Bachelor', 'Education_Level_High School', 'Education_Level_Master', 'Education_Level_PhD']


In [9]:
# Verify the in-memory demonstration; no row-level data or transformer exports.
assert len(df_preprocessed) == len(df_current) == 80_000
assert df_preprocessed.shape[1] == 36
assert df_preprocessed.isna().sum().sum() == 0
assert "Employee_ID" not in df_preprocessed.columns
assert "Hire_Date" not in df_preprocessed.columns
assert np.array_equal(df_preprocessed["Resigned"].to_numpy(), df_current["Resigned"].to_numpy())
print(f"Demonstration complete: {len(df_preprocessed):,} rows, 35 transformed predictors and one target.")
print("No processed dataset or fitted transformers exported.")
print("For model training and CV, use the member notebooks and their fold-specific preprocessing.")


Demonstration complete: 80,000 rows, 35 transformed predictors and one target.
No processed dataset or fitted transformers exported.
For model training and CV, use the member notebooks and their fold-specific preprocessing.


## Pipeline Summary & Viva Defense Guide

### Dimensionality & Progression Table

| Pipeline Stage | Sourced From | Action & Rationale | Row Count | Col Count | Net Column Change |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **0. Raw Development** | `split_registry.csv` | Isolated development partition (`partition == 'development'`) | 80,000 | 20 | Baseline |
| **1. Data Quality** | IT25100565 | Audited 0 nulls, 0 duplicates, verified unique `Employee_ID`. 0 rows dropped. | 80,000 | 20 | 0 |
| **2. Outlier Analysis** | IT24100879 | 1.5 $\times$ IQR fences computed; retained all rows to preserve workforce diversity. | 80,000 | 20 | 0 |
| **3. Categorical Encoding**| IT25100385 | OneHotEncoder expanded 4 nominal features into 23 binary dummy features. | 80,000 | 39 | $+19$ |
| **4. Feature Selection** | IT25100445 | Dropped non-predictive `Employee_ID` and redundant `Hire_Date` ($-2$). | 80,000 | 37 | $-2$ |
| **5. Standardization** | IT25100429 | StandardScaler transformed 13 numerical predictors to $\mu \approx 0, \sigma = 1.0000$. | 80,000 | 37 | 0 |
| **6. PCA Reduction** | IT25100142 | 13 standardized numerical features converted to 12 PCs (96.23% variance). | 80,000 | 36 | $-1$ |
| **Final Preprocessed** | **Consolidated** | **In-memory demonstration only** | **80,000** | **36** | **Net: +16** |

---

### What this notebook establishes

1. All six preprocessing contributions form a working demonstration, from 20 original columns to 35 transformed predictors plus the unchanged target. All development rows are retained.
2. Transformations fit only development observations. This protects the formal holdout but does not isolate development validation folds.
3. No demonstration CSV or transformers are exported. The six headline models use their own preprocessing, without adopting this PCA table as a shared input.
4. PCA retains at least 95% of numerical variance here. That describes variance preservation, not improved resignation prediction.

**Completion status:** The demonstration is complete when all substantive cells execute and the final assertions pass. It does not complete model training or final holdout evaluation.
